# Stage 1 Training — Cross-Embodiment Shared Latent Space

Yan et al. (2026) adaptation for dexterous hands.

**Setup checklist:**
- Runtime set to GPU: `Runtime > Change runtime type > T4 GPU`
- Repo synced to `MyDrive/AIST-hand/`
- `dex-urdf/` folder inside `MyDrive/AIST-hand/dex-urdf/`
- `shadow_hand_right.yaml` in `MyDrive/AIST-hand/` (root)

**Run 21** (MCP quat weight=0.5):
- `hograspnet_abl11.csv` in `MyDrive/AIST-hand/`
- `valid_robot_poses_eigengrasp_dong.npz` in `MyDrive/AIST-hand/`

**Run 22** (angle-based D_R, euler mode):
- `hograspnet_abl13.csv` in `MyDrive/AIST-hand/`
- `valid_robot_poses_eigengrasp_dong_euler.npz` in `MyDrive/AIST-hand/` (generate with precompute_dong_colab.ipynb)

## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU found. Go to Runtime > Change runtime type and select T4 GPU.')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/AIST-hand')
GITHUB_TOKEN = ''
GITHUB_USER  = 'isapedraza'
REPO_NAME    = 'AIST-hand'

# Training hyperparameters
B               = 50_000    # train batch size
N_STEPS         = 15_000    # training steps
LOG_EVERY       = 50
CKPT_EVERY      = 500
LR_WARMUP       = 500       # unused under Yan-pure constant LR (retained for backward compat)
VAL_EVERY       = 500       # eval val split every N steps
N_EVAL_BATCHES  = 20        # batches per eval pass
B_EVAL          = 5000      # batch size for eval

print(f'Drive root: {DRIVE_ROOT}')
print(f'B={B}, N_STEPS={N_STEPS}, VAL_EVERY={VAL_EVERY}')

## 3. Clone repo from GitHub

In [ ]:
REPO_DIR = f'/content/{REPO_NAME}'
BRANCH   = 'experimental'    # Run 21+: mcp D_R weight=0.5 (vs 1/sigma), seed=21266

if not os.path.exists(REPO_DIR):
    clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    os.system(f'git clone {clone_url} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} fetch origin')

os.system(f'git -C {REPO_DIR} checkout {BRANCH}')
os.system(f'git -C {REPO_DIR} pull origin {BRANCH}')
print(f'Checked out branch: {BRANCH}')

REPO_ROOT = Path(REPO_DIR)
DEX_ROOT  = DRIVE_ROOT / 'dex-urdf'
print(f'Repo: {REPO_ROOT}')
print(f'dex-urdf: {DEX_ROOT}')

## 4. Install dependencies

In [ ]:
import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION  = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'torch={TORCH_VERSION}, cuda={CUDA_VERSION}')

!pip install -q torch-geometric
!pip install -q pytorch-kinematics
!pip install -q -e /content/AIST-hand/grasp-model

print('Done.')

## 5. Run Stage 1 training

In [ ]:
import torch

# ---- Run selector: change RUN to 21 or 22 ----
RUN = 22

if RUN == 21:
    CSV_PATH         = DRIVE_ROOT / 'hograspnet_abl11.csv'
    VALID_POSES_PATH = DRIVE_ROOT / 'valid_robot_poses_eigengrasp_dong.npz'
    # Run 21: MCP D_R weight=0.5 (quat mode, abl11)
elif RUN == 22:
    CSV_PATH         = DRIVE_ROOT / 'hograspnet_abl13.csv'
    VALID_POSES_PATH = DRIVE_ROOT / 'valid_robot_poses_eigengrasp_dong_euler.npz'
    # Run 22: angle-based D_R (euler mode, abl13 + _dong_euler.npz).
    # D_R_euler uses Dong Eq.24 beta/gamma angles instead of quaternion dot products.
    # Scale risk: D_R_euler magnitudes ~5x larger than D_R_quat. w_r=1.0 first pass.
else:
    raise ValueError(f"Unknown RUN={RUN}")

URDF_PATH        = DEX_ROOT  / 'robots/hands/shadow_hand/shadow_hand_right.urdf'
HAND_CONFIG      = DRIVE_ROOT / 'shadow_hand_right.yaml'
CKPT_PATH        = DRIVE_ROOT / 'checkpoints/stage1_latest.pt'
EXTRA_HUMAN_CSV  = DRIVE_ROOT / 'data/processed/hagrid_dong.csv'

RESUME_CKPT = None

Z_DIM      = 16
SHARED_DIM = 1024
LR         = 1e-3
LAMBDA_C   = 10.0
LAMBDA_REC = 5.0
LAMBDA_LTC = 1.0
LAMBDA_TMP = 0.1
W_R        = 1.0
W_JOINTS   = 1.2
W_AHG      = 0.07
N_TRIPLETS = None
MARGIN     = 0.1
EXTRA_HUMAN_RATIO = 0.10
SEED       = 21266  # fixed -- good basin (Run 20+)

required_paths = [CSV_PATH, HAND_CONFIG, VALID_POSES_PATH, EXTRA_HUMAN_CSV]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required Drive files:\n' + '\n'.join(missing))

print(f'RUN         : {RUN}')
print(f'CSV         : {CSV_PATH}')
print(f'CKPT        : {CKPT_PATH}')
print(f'VALID_POSES : {VALID_POSES_PATH}')
print(f'HaGRID extra: {EXTRA_HUMAN_CSV} | ratio={EXTRA_HUMAN_RATIO}')
print(f'RESUME_CKPT : {RESUME_CKPT}')
print(f'Z_DIM       : {Z_DIM}  (total latent = {Z_DIM*5})')
print(f'MARGIN      : {MARGIN}')
print(f'LAMBDA_LTC  : {LAMBDA_LTC}')
print(f'W_R={W_R} | W_JOINTS={W_JOINTS} | W_AHG={W_AHG}')
print(f'Scheduler   : none (Yan-pure constant LR)')
print(f'SEED        : {SEED} (fixed)')

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "-u",
    str(REPO_ROOT / "grasp-model/src/cross_emb/train_cross_emb.py"),
    "--repo_root",        str(REPO_ROOT),
    "--dex_root",         str(DEX_ROOT),
    "--csv_path",         str(CSV_PATH),
    "--ckpt_path",        str(CKPT_PATH),
    "--hand_config",      str(HAND_CONFIG),
    "--valid_poses_path", str(VALID_POSES_PATH),
    "--extra_human_csv",  str(EXTRA_HUMAN_CSV),
    "--extra_human_ratio", str(EXTRA_HUMAN_RATIO),
    "--b",                str(B),
    "--n_steps",          str(N_STEPS),
    "--log_every",        str(LOG_EVERY),
    "--ckpt_every",       str(CKPT_EVERY),
    "--lr_warmup",        str(LR_WARMUP),
    "--z_dim",            str(Z_DIM),
    "--shared_dim",       str(SHARED_DIM),
    "--lr",               str(LR),
    "--lambda_c",         str(LAMBDA_C),
    "--lambda_rec",       str(LAMBDA_REC),
    "--lambda_ltc",       str(LAMBDA_LTC),
    "--lambda_tmp",       str(LAMBDA_TMP),
    "--w_r",              str(W_R),
    "--w_joints",         str(W_JOINTS),
    "--w_ahg",            str(W_AHG),
    "--margin",           str(MARGIN),
    "--seed",             str(SEED),
    "--val_every",        str(VAL_EVERY),
    "--n_eval_batches",   str(N_EVAL_BATCHES),
    "--b_eval",           str(B_EVAL),
    "--log_metric_stats",
]
if N_TRIPLETS is not None:
    cmd.extend(["--n_triplets", str(N_TRIPLETS)])
if RESUME_CKPT is not None:
    cmd.extend(["--resume_ckpt", str(RESUME_CKPT)])

print('Training command:')
print(' '.join(cmd))

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Training failed with code {proc.returncode}")

In [ ]:
# GPU memory diagnostics -- run after first step to check headroom
import torch
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Allocated : {allocated:.2f} GB")
print(f"Reserved  : {reserved:.2f} GB")
print(f"Total     : {total:.2f} GB")
print(f"Free      : {total - reserved:.2f} GB")